In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.profiler import profile, record_function, ProfilerActivity, schedule, tensorboard_trace_handler

BATCH_SIZE = 256
EPOCHS = 1
NUM_WORKERS = 2
LOG_DIR = "./log"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 1024),
    nn.ReLU(),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

model.train()

prof_schedule = schedule(
    wait=1,
    warmup=1,
    active=3,
    repeat=1
)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA] if torch.cuda.is_available() else [ProfilerActivity.CPU],
    schedule=prof_schedule,
    on_trace_ready=tensorboard_trace_handler(LOG_DIR),
    record_shapes=True,
    profile_memory=True,
    with_stack=True
) as prof:

    for epoch in range(EPOCHS):
        for step, (images, labels) in enumerate(train_loader):
            with record_function("data_to_device"):
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with record_function("forward_pass"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            with record_function("backward_pass"):
                loss.backward()

            with record_function("optimizer_step"):
                optimizer.step()

            if step % 10 == 0:
                print(f"step={step}, loss={loss.item():.4f}")

            prof.step()

            if step >= 19:
                break

print("\nTop operators by CUDA time:")
if torch.cuda.is_available():
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
else:
    print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))

Using device: cuda
step=0, loss=2.3009
step=10, loss=0.9383

Top operators by CUDA time:
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                               Optimizer.step#Adam.step         0.00%       0.000us         0.00%       0.000us       0.000us     291.068us       1

In [26]:
!find ./log -type f

./log/dda6d054fde3_11127.1776279583783120557.pt.trace.json
./log/dda6d054fde3_11127.1776279565791293256.pt.trace.json


In [27]:
from google.colab import files
files.download('/content/log/dda6d054fde3_11127.1776279583783120557.pt.trace.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>